# 05｜执行循环实验

[阅读路线](README.md) · [上一篇：错误处理](04-errors-and-retries.md) · [下一篇：OpenHands 源码](06-openhands-source.md)

本章总览图如下：



运行前按 README 安装依赖并填写 `.env`。主线单元会请求模型；最后的固定故障实验使用预设响应。

| 输入 | 产物 |
|---|---|
| 根目录 `notes.txt` | 运行目录内的笔记副本与回答 |
| `examples/stats.py` | 修改后的函数、差异和检查结果 |
| `.env` 中的 URL、API Key、Model | 原始请求、响应与运行状态 |

当前文件的真实 API 单元留待配置后运行；已保留的输出来自本地文件检查和固定故障实验。


## 1. 环境准备

下面单元定位章节目录，打印 `notes.txt`。标准输出的第一行应与文件内容一致；修改文件后重跑本格即可核对。


In [1]:
from pathlib import Path
import sys
import json

candidates = [Path.cwd(), *list(Path.cwd().parents)[:3]]
ROOT = next((p for base in candidates for p in (base, base / "03-agent-loop")
             if (p / "notes.txt").is_file() and (p / "code" / "live_run.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从完整章节目录打开 Notebook。")
sys.path.insert(0, str(ROOT / "code"))
from live_run import run_stage
from build_report import build_report

def table(headers, rows):
    def safe(x):
        return str(x).replace("|", r"\|").replace("\n", " ")
    print("| " + " | ".join(headers) + " |")
    print("| " + " | ".join("---" for _ in headers) + " |")
    for row in rows:
        print("| " + " | ".join(safe(x) for x in row) + " |")

print((ROOT / "notes.txt").read_text(encoding="utf-8"), end="")
print("输入文件已找到：notes.txt、examples/stats.py")

本周完成了工具接入与循环日志。
输入文件已找到：notes.txt、examples/stats.py


In [ ]:
from config import load_settings
settings = load_settings()
print("URL:", settings.base_url)
print("Model:", settings.model)
print("API Key: 已读取")

## 2. 模型调用

这一格运行 v0。它打印回答和产物目录，保存 `requests.jsonl`、`responses.jsonl` 与 `answer.md`。

| 检查点 | 预期 |
|---|---|
| 控制台 `model_calls` | 1 |
| `answer.md` | 一句关于算术平均值的说明 |
| `responses.jsonl` | 服务返回的完整响应 |


In [ ]:
from v0_model_call import main as run_v0
run_v0()

## 3. 工具结果回传

运行 v1 后，读取保存的 API 请求，找出其中的 tool 消息。应能看到本次笔记的内容，以及与请求对应的 `tool_call_id`。


In [ ]:
result1, directory1 = run_stage("v1")
requests1 = [json.loads(line)["request"] for line in (directory1 / "requests.jsonl").read_text(encoding="utf-8").splitlines()]
rows = []
for attempt, request in enumerate(requests1, 1):
    for message in request["messages"]:
        if message["role"] == "tool":
            rows.append([attempt, message["tool_call_id"], message["content"]])
table(["第几次请求", "工具调用 ID", "模型收到的工具结果"], rows)
print("最终回答：")
print((directory1 / "answer.md").read_text(encoding="utf-8"))
if not rows:
    print("本次未产生工具结果，请检查 responses.jsonl。")

## 4. 工具与预算对照

先运行 v2，再让 v3 分别使用 2 次与 12 次调用预算。三次运行使用同一份初始函数，保存到不同目录。

预期得到三条独立记录。实际检查是否通过取决于各次模型输出，直接查看表中的结果。


In [ ]:
result2, directory2 = run_stage("v2")
limited, limited_dir = run_stage("v3", max_steps=2)
regular, regular_dir = run_stage("v3", max_steps=12)
table(["运行", "退出原因", "调用次数", "最终检查"], [
    ["v2", result2["reason"], result2["model_calls"], result2["acceptance"]["passed"]],
    ["v3 / 2 次预算", limited["reason"], limited["model_calls"], limited["acceptance"]["passed"]],
    ["v3 / 12 次预算", regular["reason"], regular["model_calls"], regular["acceptance"]["passed"]],
])
print("v3 的修改：")
print((regular_dir / "changes.diff").read_text(encoding="utf-8"))

## 5. 错误处理

运行 v4，读取 `trace.jsonl` 中的错误事件，再查看最终检查表。如果本次没有发生错误，错误表为空。


In [ ]:
result4, directory4 = run_stage("v4")
errors = [event for event in result4["trace"]
          if event["event"] in ("model_error", "protocol_error", "empty_response", "run_error")
          or (event["event"] == "tool_result" and not event["observation"]["ok"])]
print(json.dumps(errors, ensure_ascii=False, indent=2))
table(["检查项", "通过", "详情"], [[c["name"], c["passed"], c["detail"]]
      for c in result4["acceptance"]["checks"]])
print("本次记录：", directory4 / "report.md")

## 6. 结果汇总

这一步不再调用模型。它读取 `runs/` 下已有的 `result.json`，生成 `comparison.md` 和 `comparison.json`。

标准输出的第一行是 `runs=<读取到的运行数>`，第二行是保存路径。


In [ ]:
rows = build_report()
table(["阶段", "退出原因", "调用次数", "检查通过"],
      [[r["stage"], r["reason"], r["model_calls"], r["passed"]] for r in rows])

## 7. 故障实验

下面只使用预设响应，分别制造提前结束、预算耗尽、连续空响应和旧测试结果过期。它们用于观察控制逻辑，输出可重复。

| 场景 | 预期原因 | 预期检查 |
|---|---|---|
| `early_finish` | `finish` | False |
| `step_limit` | `step_limit` | True |
| `empty_response` | `empty_response_limit` | False |
| `stale_test` | `finish` | False |


In [2]:
from scenarios import run_case
fixed_results = [run_case(name) for name in ("early_finish", "step_limit", "empty_response", "stale_test")]
table(["固定场景", "退出原因", "调用次数", "最终检查"],
      [[r["case"], r["reason"], r["model_calls"], r["acceptance"]["passed"]] for r in fixed_results])
assert [r["acceptance"]["passed"] for r in fixed_results] == [False, True, False, False]

| 固定场景 | 退出原因 | 调用次数 | 最终检查 |
| --- | --- | --- | --- |
| early_finish | finish | 1 | False |
| step_limit | step_limit | 2 | True |
| empty_response | empty_response_limit | 2 | False |
| stale_test | finish | 5 | False |


## 8. 输入修改实验

1. 修改根目录 `notes.txt`，重新运行第 3 节，比较两次的 tool 内容和回答。
2. 将第 4 节预算改成 1、3、6，比较退出原因、调用次数与代码差异。
3. 打开某次 `report.md`，用 `requests.jsonl` 和 `trace.jsonl` 找到支撑其结论的操作。
4. 继续阅读 [06｜OpenHands 源码](06-openhands-source.md)，查找对应的循环、动作与结果事件。
